In [1]:
print("hello world")

hello world


In [ ]:
import openai
import speech_recognition as sr
from dotenv import load_dotenv
from scipy.io.wavfile import write
import os

# 1. 초기 설정 (API 키 설정 필요)
load_dotenv()
client = openai.OpenAI()

def record_audio():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print('말씀하세요.')
        # return ("햄버거")
        audio = recognizer.listen(source, timeout=10, phrase_time_limit=10)
        text = recognizer.recognize_google(audio, language='ko-KR')
        return (text)

        
def stt_whisper(filename="stt_input.wav"):
    with open(filename, "rb") as f:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", 
            file=f
        )
    return transcript.text

def ask_llm(user_text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"너는 요리 보조 AI야. 짧고 효율적이고 명확하게 답해줘. 예시 레시피) 김치볶음밥: 1. 양파 볶기 2. 김치 넣기 3. 밥과 고추장 넣고 볶기"},
            {"role": "user", "content": user_text}
        ]
    )
    return response.choices[0].message.content

def tts_speak(input_text):
    with client.audio.speech.with_streaming_response.create(
        model='gpt-4o-mini-tts',
        voice='shimmer',
        input=input_text
    ) as response:
        response.stream_to_file('tts_output.mp3')

    os.system("start tts_output.mp3") # Windows 기준


max_call = 2

for _ in range(max_call):
    # 1. 음성 입력 받기
    user_input = record_audio()
    print(f"사용자: {user_input}")
    # # 2. STT 적용
    # file_name = "stt_input.wav"
    # user_input = stt_whisper(file_name)
    
    if "종료" in user_input: break

    # 3. LLM 요청 (프롬프트 엔지니어링)
    ai_answer = ask_llm(user_input)
    
    # 4. 결과 출력 및 TTS 재생
    print(f"응답: {ai_answer}")
    tts_speak(ai_answer)
    

말씀하세요.
사용자: 햄버거
응답: 햄버거: 1. 패티 굽기 2. 빵 반쪽 구우기 3. 상추, 토마토, 피클 올리기 4. 패티 얹고 치즈 올리기 5. 다른 빵 반쪽 덮기
말씀하세요.
사용자: 햄버거
응답: 햄버거: 1. 패티 소금, 후추로 간 후 굽기 2. 빵 굽기 3. 양상추, 토마토, 양파 준비 4. 빵에 소스 바르고 재료 쌓기 5. 패티 올리고 뚜껑 닫기
